# Computational Geometry

A small, dependable toolkit for problems on points and polygons in the plane — orientation, segment intersection, convex hull, and point-in-polygon — built from a single primitive: the **2D cross product**. The angle this notebook takes: geometry is where floating-point quietly betrays you, so we lean on **exact integer arithmetic** wherever the inputs allow, and prove every routine against hand-checkable inputs.

**Contents**

1. **Primitives** — points, vectors, and the cross product (orientation)
2. **Segment intersection** — the four-orientation rule
3. **Convex hull** — Andrew's monotone chain, O(n log n)
4. **Point in polygon** — ray casting (parity)
5. **Shoelace area & closest pair** — a bonus and a pointer
6. **When to use what**
7. **Cheat-sheet & robustness notes**

## 1. Primitives — points, vectors, and the cross product

A **point** is just a tuple `(x, y)`; a **vector** is the difference of two points. Almost every planar algorithm reduces to one question — *which way do we turn?* — answered by the **2D cross product**.

For vectors **u** = (uₓ, u_y) and **v** = (vₓ, v_y), the (scalar) cross product is

```
u × v = uₓ·v_y − u_y·vₓ
```

It equals the **signed area** of the parallelogram they span. Applied to three points O, A, B as `cross(O, A, B) = (A−O) × (B−O)`, its **sign** tells you the orientation of the turn O→A→B:

| sign | turn | meaning |
|:---:|---|---|
| `> 0` | counter-clockwise (left) | B is left of ray O→A |
| `< 0` | clockwise (right) | B is right of ray O→A |
| `= 0` | none | O, A, B are **collinear** |

That single sign is the workhorse for orientation, intersection, hulls, and area.

In [1]:
def cross(o, a, b):
    """(a - o) x (b - o): signed area; >0 left turn, <0 right turn, 0 collinear."""
    return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

def orient(o, a, b):
    """Sign of the cross product: +1 ccw/left, -1 cw/right, 0 collinear."""
    c = cross(o, a, b)
    return (c > 0) - (c < 0)

#        B(0,4)            orientation of O -> A -> B
#        |
#  O(0,0)--A(4,0)
assert orient((0, 0), (4, 0), (0, 4)) ==  1   # turn left (counter-clockwise)
assert orient((0, 0), (0, 4), (4, 0)) == -1   # turn right (clockwise)
assert orient((0, 0), (2, 2), (5, 5)) ==  0   # straight on -> collinear

print("ccw  :", orient((0, 0), (4, 0), (0, 4)))
print("cw   :", orient((0, 0), (0, 4), (4, 0)))
print("colin:", orient((0, 0), (2, 2), (5, 5)))
print("signed area of parallelogram (3,0),(0,2):", cross((0, 0), (3, 0), (0, 2)))

ccw  : 1
cw   : -1
colin: 0
signed area of parallelogram (3,0),(0,2): 6


### Why integers matter — float `==` is a trap

The cross product is the one place geometry code most often breaks: a collinearity test is `cross(...) == 0`, and **floats almost never land exactly on zero**. Three points that are *truly* collinear can produce a tiny nonzero float because the subtraction and multiplication can't be represented exactly in binary.

The three points below lie on a perfect line (x steps +0.1, y steps −0.1), yet the float cross product is a nonzero crumb — so a float `== 0` test wrongly reports "not collinear." The *same points scaled to integers* give an exact `0`. **Keep coordinates integer when you can** (scale rationals up), and the cross product is exact and total.

In [2]:
# Three genuinely collinear points (x: 0.43, 0.53, 0.63 ; y: 0.84, 0.74, 0.64)
A, B, C = (0.43, 0.84), (0.53, 0.74), (0.63, 0.64)
fcross = (B[0] - A[0]) * (C[1] - A[1]) - (B[1] - A[1]) * (C[0] - A[0])
print("float cross:", fcross, " -> '== 0'?", fcross == 0.0)   # tiny nonzero crumb!
assert fcross != 0.0          # float WRONGLY says: not collinear

# Same geometry in integers (multiply every coordinate by 100):
Ai, Bi, Ci = (43, 84), (53, 74), (63, 64)
icross = (Bi[0] - Ai[0]) * (Ci[1] - Ai[1]) - (Bi[1] - Ai[1]) * (Ci[0] - Ai[0])
print("int   cross:", icross, " -> '== 0'?", icross == 0)
assert icross == 0            # integers: exactly collinear, no rounding

print("\nLesson: with float coords, the crumb is ~%.1e, not 0 -> never test 'cross == 0' on floats."
      % abs(fcross))

float cross: -6.938893903907228e-18  -> '== 0'? False
int   cross: 0  -> '== 0'? True

Lesson: with float coords, the crumb is ~6.9e-18, not 0 -> never test 'cross == 0' on floats.


## 2. Segment intersection — the four-orientation rule

Do segments **p₁p₂** and **p₃p₄** intersect? The robust test uses four orientations and no division (so it stays exact on integers):

1. Compute `d1 = orient(p3, p4, p1)`, `d2 = orient(p3, p4, p2)` — which sides of line p₃p₄ the endpoints of the first segment fall on.
2. Compute `d3 = orient(p1, p2, p3)`, `d4 = orient(p1, p2, p4)` — symmetrically.
3. **General (proper) case:** if `d1 ≠ d2` *and* `d3 ≠ d4`, each segment straddles the other's line — they cross at one interior point.
4. **Collinear / touching cases:** if any orientation is `0`, the points are collinear; the segments meet only if the collinear endpoint actually lies *on* the other segment — an **on-segment** bounding-box check.

This catches every case: proper crossings, endpoint touches (T-junctions and shared endpoints), collinear overlap, and collinear-but-disjoint.

In [3]:
def on_segment(p, q, r):
    """Given q is collinear with segment p-r, is q within the segment's bounding box?"""
    return (min(p[0], r[0]) <= q[0] <= max(p[0], r[0]) and
            min(p[1], r[1]) <= q[1] <= max(p[1], r[1]))

def segments_intersect(p1, p2, p3, p4):
    d1 = orient(p3, p4, p1)
    d2 = orient(p3, p4, p2)
    d3 = orient(p1, p2, p3)
    d4 = orient(p1, p2, p4)
    if d1 != d2 and d3 != d4:               # proper crossing: each straddles the other
        return True
    # collinear / touching: a zero orientation must also lie ON the segment
    if d1 == 0 and on_segment(p3, p1, p4): return True
    if d2 == 0 and on_segment(p3, p2, p4): return True
    if d3 == 0 and on_segment(p1, p3, p2): return True
    if d4 == 0 and on_segment(p1, p4, p2): return True
    return False

# proper X-crossing                 #  \ /
assert segments_intersect((0, 0), (4, 4), (0, 4), (4, 0)) is True    #   X
# parallel, apart                                                    #  / \
assert segments_intersect((0, 0), (1, 0), (0, 1), (1, 1)) is False
# collinear but disjoint (gap on the same line)
assert segments_intersect((0, 0), (1, 1), (2, 2), (3, 3)) is False
# collinear OVERLAPPING on the x-axis ([0,4] and [2,6] share [2,4])
assert segments_intersect((0, 0), (4, 0), (2, 0), (6, 0)) is True
# T-junction: second segment's endpoint touches the middle of the first
assert segments_intersect((0, 0), (4, 0), (2, 0), (2, 4)) is True
# shared endpoint (the two segments meet at exactly (2,2))
assert segments_intersect((0, 0), (2, 2), (2, 2), (4, 0)) is True

print("proper cross   :", segments_intersect((0, 0), (4, 4), (0, 4), (4, 0)))
print("parallel apart :", segments_intersect((0, 0), (1, 0), (0, 1), (1, 1)))
print("collinear gap  :", segments_intersect((0, 0), (1, 1), (2, 2), (3, 3)))
print("collinear over :", segments_intersect((0, 0), (4, 0), (2, 0), (6, 0)))
print("T-junction     :", segments_intersect((0, 0), (4, 0), (2, 0), (2, 4)))
print("shared endpoint:", segments_intersect((0, 0), (2, 2), (2, 2), (4, 0)))

proper cross   : True
parallel apart : False
collinear gap  : False
collinear over : True
T-junction     : True
shared endpoint: True


The on-segment branch is what separates a **collinear overlap** (segments share a stretch) from a **collinear gap** (same line, but disjoint) — without it, every collinear pair would falsely "intersect." Because the whole test is built from `orient` (only `*`, `-`, comparisons) it is **exact on integer coordinates**: no division, no epsilon, no float `==`. The companion problem of intersecting *many* segments at once is the **sweep line** pattern.

## 3. Convex hull — Andrew's monotone chain, O(n log n)

The **convex hull** is the smallest convex polygon containing all the points — the shape a rubber band snaps to. **Andrew's monotone chain** computes it in **O(n log n)**, dominated by one sort (the **sorting** notebook's Timsort), then two linear passes:

1. **Sort** the points by `(x, y)`.
2. Build the **lower hull** left→right: push each point, and while the last three make a non-left (clockwise or collinear) turn, pop the middle one.
3. Build the **upper hull** the same way over the reversed order.
4. Concatenate, dropping each chain's last point (it's the other chain's first).

The turn test is exactly `cross(...) <= 0` → pop. Using `<= 0` (not `< 0`) discards points lying *on* a hull edge, yielding the **minimal** hull of true corners. Output is counter-clockwise, starting from the bottom-left.

In [4]:
def convex_hull(points):
    pts = sorted(set(points))           # sort by (x, y); dedupe
    if len(pts) <= 1:
        return pts

    def build(seq):
        chain = []
        for p in seq:
            # pop while the last turn is clockwise or collinear (cross <= 0)
            while len(chain) >= 2 and cross(chain[-2], chain[-1], p) <= 0:
                chain.pop()
            chain.append(p)
        return chain

    lower = build(pts)                  # left -> right
    upper = build(reversed(pts))        # right -> left
    return lower[:-1] + upper[:-1]      # drop shared endpoints -> CCW hull

# Known set: a 4x4 square's corners + an interior point + 3 collinear points on the bottom edge.
P = [(0, 0), (4, 0), (4, 4), (0, 4),    # the true corners
     (2, 2),                            # interior  -> must be dropped
     (1, 0), (2, 0), (3, 0)]            # on bottom edge -> must be dropped
hull = convex_hull(P)
print("hull (CCW):", hull)
assert hull == [(0, 0), (4, 0), (4, 4), (0, 4)]     # exactly the 4 corners

# Independent check: brute-force gift-wrapping (skip collinear, keep farthest).
def gift_wrap(points):
    pts = list(set(points))
    start = min(pts)
    out, p = [], start
    while True:
        out.append(p)
        q = pts[0] if pts[0] != p else pts[1]
        for r in pts:
            if r == p:
                continue
            o = orient(p, q, r)
            far = (r[0]-p[0])**2 + (r[1]-p[1])**2 > (q[0]-p[0])**2 + (q[1]-p[1])**2
            if o < 0 or (o == 0 and far):
                q = r
        p = q
        if p == start:
            break
    return out

assert set(convex_hull(P)) == set(gift_wrap(P))     # two algorithms, same corner set
print("matches independent gift-wrapping:", set(hull) == set(gift_wrap(P)))

hull (CCW): [(0, 0), (4, 0), (4, 4), (0, 4)]
matches independent gift-wrapping: True


Two structurally different algorithms (sort-and-scan vs. wrap-around-the-outside) agree on the corner set — strong evidence the hull is right. The collinear edge points `(1,0),(2,0),(3,0)` and the interior `(2,2)` were correctly dropped because `cross <= 0` removes any point that doesn't make a strict left turn. (Switch to `< 0` if you *want* to keep collinear edge points.) The sort makes it **O(n log n)**; everything after is two linear passes with amortized-O(1) pops.

## 4. Point in polygon — ray casting (parity)

Is a point inside an arbitrary (possibly non-convex) polygon? **Ray casting**: shoot a ray from the point (here, straight to the right along +x) and count how many polygon edges it crosses. **Odd ⇒ inside, even ⇒ outside** — the Jordan curve theorem made practical.

The implementation walks each edge `(i, j)` and asks:

- Does the edge **straddle** the horizontal line `y = pt.y`? Tested as `(y_i > y) != (y_j > y)` — a half-open rule that counts each crossing once and dodges double-counting at shared vertices.
- If so, compute the x of the crossing; if it's to the **right** of the point, toggle the inside/outside flag.

It runs in **O(n)** for an n-vertex polygon and needs no convexity.

In [5]:
def point_in_polygon(poly, pt):
    """Ray casting (even-odd rule). poly: list of (x, y) vertices in order."""
    x, y = pt
    n = len(poly)
    inside = False
    j = n - 1                                  # previous vertex (wraps around)
    for i in range(n):
        xi, yi = poly[i]
        xj, yj = poly[j]
        if (yi > y) != (yj > y):               # edge straddles the horizontal ray
            x_cross = (xj - xi) * (y - yi) / (yj - yi) + xi
            if x < x_cross:                    # crossing is to the right of pt
                inside = not inside            # toggle parity
        j = i
    return inside

#  A non-convex "C" (opens to the right):
#   (0,6)---------(6,6)
#     |   (1,5)     |
#     |          (6,4)
#     |    (1,3)  (2,4)
#     |          (2,2)
#     |   (1,1)  (6,2)
#   (0,0)---------(6,0)
poly = [(0, 0), (6, 0), (6, 2), (2, 2), (2, 4), (6, 4), (6, 6), (0, 6)]

inside  = [(1, 3), (1, 1), (1, 5)]             # clearly within the left bar of the C
outside = [(4, 3), (7, 3), (3, 3), (-1, 3)]    # in the notch, right of, and beyond the polygon
for p in inside:
    assert point_in_polygon(poly, p) is True,  p
for p in outside:
    assert point_in_polygon(poly, p) is False, p

print("inside  :", {p: point_in_polygon(poly, p) for p in inside})
print("outside :", {p: point_in_polygon(poly, p) for p in outside})

inside  : {(1, 3): True, (1, 1): True, (1, 5): True}
outside : {(4, 3): False, (7, 3): False, (3, 3): False, (-1, 3): False}


The point `(4, 3)` sits in the **notch** of the C — visually "surrounded" on three sides but genuinely outside — and the parity rule gets it right, which a naive bounding-box test would not. **Boundary points are a definitional gray area**: a point exactly *on* an edge may read inside or outside depending on the half-open `>` comparisons, so if on-edge membership matters, test it explicitly with a collinear + on-segment check (§2) before ray casting. The **winding number** method is an alternative that also handles self-intersecting polygons.

## 5. Shoelace area & a closest-pair pointer

The **shoelace formula** (Gauss) gives a simple polygon's area by summing cross products of consecutive vertices:

```
Area = ½ · | Σ (xᵢ·yᵢ₊₁ − xᵢ₊₁·yᵢ) |
```

It's **O(n)**, and on integer coordinates the *signed* sum is an exact integer — twice the area — whose **sign also reveals orientation** (positive ⇒ counter-clockwise, negative ⇒ clockwise). Below, the C-shape's area is the 6×6 bounding box (36) minus its 4×2 notch (8) = **28**.

In [6]:
def polygon_area(poly):
    """Shoelace: absolute area of a simple polygon. Exact on integer coords."""
    n = len(poly)
    s = 0
    for i in range(n):
        x1, y1 = poly[i]
        x2, y2 = poly[(i + 1) % n]
        s += x1 * y2 - x2 * y1          # sum of consecutive cross products
    return abs(s) / 2                   # |signed area|; sign of s gives orientation

def signed_area2(poly):                 # twice the signed area (exact int); sign = orientation
    return sum(poly[i][0] * poly[(i+1) % len(poly)][1] - poly[(i+1) % len(poly)][0] * poly[i][1]
               for i in range(len(poly)))

assert polygon_area([(0, 0), (4, 0), (4, 4), (0, 4)]) == 16     # 4x4 square
assert polygon_area([(0, 0), (4, 0), (0, 3)]) == 6             # right triangle, legs 4 and 3
poly = [(0, 0), (6, 0), (6, 2), (2, 2), (2, 4), (6, 4), (6, 6), (0, 6)]
assert polygon_area(poly) == 28                                # 36 (box) - 8 (notch)

print("square area :", polygon_area([(0, 0), (4, 0), (4, 4), (0, 4)]))
print("triangle    :", polygon_area([(0, 0), (4, 0), (0, 3)]))
print("C-shape     :", polygon_area(poly))
print("orientation : CCW square signed-2A =", signed_area2([(0, 0), (4, 0), (4, 4), (0, 4)]),
      "(>0 -> CCW) | CW =", signed_area2([(0, 0), (0, 4), (4, 4), (4, 0)]), "(<0 -> CW)")

square area : 16.0
triangle    : 6.0
C-shape     : 28.0
orientation : CCW square signed-2A = 32 (>0 -> CCW) | CW = -32 (<0 -> CW)


**Closest pair of points** is the other classic of the field and doesn't fit on one cross product. The brute-force check is O(n²); the textbook **O(n log n)** solution is a **divide & conquer** that splits by x, recurses, then merges across the dividing strip (a relative of the **sweep line** pattern — sort the points and sweep, keeping a small candidate window). It's the canonical example of geometric divide-and-conquer, in the spirit of the **sorting** notebook's merge sort.

## 6. When to use what

| You need... | Reach for | Why |
|---|---|---|
| Left / right / collinear of a turn | `orient(o, a, b)` (sign of cross) | one primitive; exact on ints |
| Do two segments cross? | four-orientation rule + `on_segment` | handles touch & collinear cases, no division |
| Smallest enclosing convex polygon | Andrew's monotone chain | O(n log n), simple, stable corners |
| Is a point inside a polygon? | ray casting (even-odd) | O(n), works on non-convex shapes |
| Area / orientation of a polygon | shoelace formula | O(n), exact signed area on ints |
| Closest pair of points | divide & conquer / sweep | O(n log n) vs. O(n²) brute force |
| Intersect *many* segments | sweep line | event-driven, far better than all-pairs |

**Rule of thumb:** if every routine you need is built from `orient`, keep coordinates **integer** and you get exact, branch-clean answers. Reach for floats only when the geometry is inherently continuous (e.g. the actual intersection *point*), and even then compute the *decision* (does it cross?) in integers first.

## 7. Cheat-sheet & robustness notes

**The one primitive:** `cross(o, a, b) = (a−o) × (b−o)`. Its sign is orientation; its magnitude is twice a triangle's area. Everything else is built on it.

| Operation | Algorithm | Complexity | Exact on ints? |
|---|---|:---:|:---:|
| Orientation of 3 points | sign of cross product | O(1) | ✅ |
| Segment intersection (yes/no) | four-orientation rule | O(1) | ✅ |
| Convex hull | Andrew's monotone chain | O(n log n) | ✅ (sort + cross) |
| Point in polygon | ray casting (even-odd) | O(n) | ⚠️ uses division for x_cross |
| Polygon area / orientation | shoelace formula | O(n) | ✅ (signed sum is `2·area`) |
| Closest pair of points | divide & conquer / sweep | O(n log n) | ✅ via squared distances |

**Robustness — the geometry-specific gotchas:**

- **Use integers; avoid float `==`.** A collinearity test is `cross == 0`; floats rarely hit exactly 0 (§1). Scale rational inputs up to integers so the cross product is exact.
- **Compare squared distances**, never `math.sqrt`, when you only need to rank or test distances — `sqrt` is both slower and another source of float error.
- **Avoid division until the end.** The whole intersection/hull/orientation stack needs only `*`, `−`, and comparisons. Divide only to produce an actual coordinate (the crossing point, `x_cross`), and accept it's then floating-point.
- **Decide the boundary policy.** "On an edge / vertex" is in-or-out by convention; ray casting's half-open `>` rule picks one, but state your choice and test it explicitly when it matters.